###上旧下新

In [2]:
from pecanpy import pecanpy
import numpy as np
import pandas as pd
from tqdm import tqdm


In [6]:
# 先生成 edgelist 文件
df = pd.read_csv('../data/kg.csv')
df['src'] = df['x_type'] + '::' + df['x_index'].astype(str)
df['dst'] = df['y_type'] + '::' + df['y_index'].astype(str)
df[['src', 'dst']].to_csv('../data/kg_edgelist.txt',
                           sep='\t', index=False, header=False)

# 再接着跑 pecanpy
g = pecanpy.SparseOTF(p=1, q=1, workers=8, verbose=True)
g.read_edg('../data/kg_edgelist.txt', weighted=False, directed=False)

/tmp/ipykernel_58636/1813317598.py:2: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/kg.csv')


In [12]:

g = pecanpy.SparseOTF(p=1, q=1, workers=8, verbose=True)
g.read_edg('../data/kg_edgelist.txt', weighted=False, directed=False)

model = g.embed(
    dim=128,
    num_walks=10,
    walk_length=30,
    window_size=10,
    epochs=1,
    verbose=True  # pecanpy 自带进度条，已覆盖随机游走和训练两个阶段
)



  0%|          | 0/1293750 [00:00<?, ?it/s]

Took 00:03:53.21 to generate walks
Took 00:01:26.40 to train embeddings


In [18]:
import pickle
nodes = g.nodes
# 关键：np.array() 会重建数组，清除 pecanpy 留下的旧格式标记
records = [{'id': node, 'embedding': np.array(emb, dtype=np.float32).copy()}
           for node, emb in zip(nodes, model)]

pd.DataFrame(records).to_pickle('../data/node2vec_embeddings.pkl')

                   id                                          embedding
0     gene/protein::0  [-0.053788982, -0.31126612, 0.3648416, 0.02539...
1  gene/protein::8889  [0.03155777, -0.3213998, 0.3851969, 0.03080304...
2     gene/protein::1  [0.11723068, -0.31640664, 0.15420897, -0.05792...
3  gene/protein::2798  [0.012343271, -0.17480423, 0.43843645, 0.03746...
4     gene/protein::2  [-0.05256521, -0.13608967, 0.17404935, -0.0144...


In [7]:
node2vec_embeddings = pd.read_pickle('../data/node2vec_embeddings.pkl')
pubmedbert_embeddings = pd.read_pickle('../data/pubmedbert_embeddings.pkl')

In [8]:
ent_dim = len(node2vec_embeddings.iloc[0]['embedding'])
print(f"node2vec_embeddings 维度: {ent_dim}")
ent_dim = len(pubmedbert_embeddings.iloc[0]['embedding'])
print(f"pubmedbert_embeddings 维度: {ent_dim}")

node2vec_embeddings 维度: 128
pubmedbert_embeddings 维度: 768
